# Plot results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

Specify the path to the folder where the results are stored.

In [ ]:
input_path = '/home/timo/CLionProjects/PSOMAB/cmake-build-debug/'

In [ ]:
visited = pd.DataFrame()

## Density Plot (Heatmap)

Functions to read the output files of the algorithms.

In [ ]:
def read_memory(path: str):
    df = pd.read_csv(path, header=None)
    df.drop(df.columns[-1], axis=1, inplace=True)
    return df

In [ ]:
def get_file_list():
    file_list = []
    for file in os.listdir(input_path):
        if file.startswith('ackley_psomab_memory'):
            file_list.append(file)
    return file_list

In [ ]:
def aggregate_memory(files):
    df = pd.DataFrame()
    for file in files:
        path = input_path + file  #Add path
        df_temp = read_memory(path)
        df = df.add(df_temp, fill_value=0)
    return df

Read from files, aggregate the results and apply logarithmic scaling.

In [ ]:
visited = aggregate_memory(get_file_list())

In [ ]:
visited = np.where(visited > 0, np.log10(visited), visited)

Generate heatmap.

In [ ]:
sns.set(rc = {'figure.figsize':(12,12)})
fig, ax = plt.subplots()
ax.set_xlabel('x')
ax.set_title('Visited Solutions')
ax.set_ylabel('y')
sns.heatmap(visited, xticklabels=100, yticklabels=100, ax=ax)

## Optimality Gap (Boxplots, Convergence Plots)

Functions to read the output files of the algorithms.

In [ ]:
def read_csv(path: str, save_after: int):
    df = pd.read_csv(path, header=None)
    df.drop(df.columns[-1], axis=1, inplace=True)
    df = df.T
    df = df.reindex(df.index.repeat(save_after)).reset_index(drop=True)
    return df

In [ ]:
def get_optimality_gap(df: pd.DataFrame, optimum: float):
    return df.mean(axis=1) - optimum

In [ ]:
# get the newest file from folder which starts with two strings
def get_file(problem: str, algo: str):
    file_list = []
    for file in os.listdir(input_path):  #Add path
        if file.startswith(problem + '_' + algo + '_1'):
            file_list.append(file)
    file_list.sort()
    return file_list[-1]

Read from files, calculate mean and then calculate optimality gap.

In [ ]:
algorithms = ['psomab', 'lapso', 'psoocbaa', 'pso', 'psogd', 'psoern']

In [ ]:
problem = 'ackley'

In [ ]:
optimality_gaps = pd.DataFrame()

In [ ]:
dataframes = []

In [ ]:
for algorithm in algorithms:
    df = read_csv(input_path + get_file(problem, algorithm), 50)
    optimality_gaps[algorithm] = get_optimality_gap(df, 0)
    dataframes.append(df)

Generate Convergence Plot.

In [ ]:
sns.set(rc = {'figure.figsize':(12,12)})
sns.set_style('whitegrid')
fig, ax = plt.subplots()
ax.set_xlabel('Evaluations of the objective function')
ax.set_title('Optimality gap of Algorithms on the Ackley function')
ax.set_ylabel('Optimality gap')
sns.lineplot(data=optimality_gaps, ax=ax)

In [ ]:
def get_best_values(df: pd.DataFrame, optimum: float):
    return df.iloc[-1] - optimum

In [ ]:
best_values = pd.DataFrame()

In [ ]:
for algorithm in algorithms:
    best_values[algorithm] = get_best_values(dataframes[algorithms.index(algorithm)], 0)

Generate boxplots.

In [ ]:
sns.set(rc = {'figure.figsize':(12,12)})
sns.set_style('whitegrid')
_, ax = plt.subplots()
ax.set_xlabel('Algorithms')
ax.set_title('Optimality gap of Algorithms on the Ackley function')
ax.set_ylabel('Optimality gap')
sns.boxplot(data=best_values, ax=ax)